# Maximum Likelihood

机器学习训练的本质是"选一个分布使数据最可能"。本课建立极大似然估计（MLE）的完整体系，并揭示一个关键事实：**所有常用损失函数都是负对数似然**。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import torch
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 似然函数：数据在参数下的概率


设样本 $x_1, \dots, x_n$ 独立同分布（i.i.d.）于 $p(x;\theta)$。**似然**是把参数当成变量、把数据当成固定的函数：

$$L(\theta) = \prod_{i=1}^n p(x_i; \theta)$$

MLE：找使 $L(\theta)$ 最大的 $\hat\theta$。"数据最可能的参数"就是我们要的。


## 2. 对数似然：乘积变求和


$n$ 个 $\in (0,1)$ 的数相乘会下溢到 0（数值灾难）。取对数把乘积变求和：

$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^n \log p(x_i; \theta)$$

对数单调，最大值点不变。**最大对数似然 ≡ 最小负对数似然（NLL）**——NLL 就是机器学习的损失函数。


In [ ]:
xs = np.linspace(0.01, 0.99, 100)
# 展示下溢：n 个 0.5 连乘
n = np.arange(1, 1200)
Lvals = 0.5**n
print("0.5 连乘 1000 次:", Lvals[-1], "→ 下溢为 0")
print("对数似然:", n[-1]*np.log(0.5), "→ 正常有限值")
plt.figure(figsize=(8, 3.5))
plt.semilogy(n, Lvals)
plt.xlabel('n（样本数）'); plt.ylabel('L(θ)')
plt.title('原始似然随样本数下溢 → 必须用对数似然')


## 3. MLE 的解析推导：高斯与伯努利


**高斯 $N(\mu,\sigma^2)$**：对数似然对 $\mu, \sigma^2$ 求导置零，得到

$$\hat\mu = \bar{x} = \frac{1}{n}\sum x_i, \qquad \hat\sigma^2 = \frac{1}{n}\sum (x_i - \bar{x})^2$$

**伯努利 $\mathrm{Bern}(p)$**：$\hat p = \frac{1}{n}\sum x_i$（出现频率）。

完整推导（高斯 $\mu$）：$\frac{\partial}{\partial\mu}\sum -\frac{(x_i-\mu)^2}{2\sigma^2} = \sum\frac{x_i-\mu}{\sigma^2} = 0 \Rightarrow \mu = \bar{x}$。


In [ ]:
rng = np.random.default_rng(0)
data = rng.normal(1.7, 0.4, 500)

# 解析 MLE
mu_mle = data.mean()
sigma_mle = np.sqrt(((data - mu_mle)**2).mean())
print(f"解析 MLE: μ̂ = {mu_mle:.4f}, σ̂ = {sigma_mle:.4f}（真值 1.7, 0.4）")


In [ ]:
# 数值 MLE：用梯度下降最小化 NLL（torch 自动求导），验证解析解
t = torch.tensor(data, dtype=torch.float64)
mu = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
log_sigma = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([mu, log_sigma], lr=0.05)

for step in range(400):
    opt.zero_grad()
    s = torch.exp(log_sigma)
    # 高斯 NLL: 0.5·log(2π) + log σ + (x-μ)²/(2σ²)
    nll = (0.5*np.log(2*np.pi) + log_sigma + (t - mu)**2 / (2*s**2)).mean()
    nll.backward()
    opt.step()

print(f"数值 MLE: μ̂ = {mu.item():.4f}, σ̂ = {torch.exp(log_sigma).item():.4f}")
print("与解析解一致 ✓")


## 4. 损失函数 = 负对数似然


这是机器学习与概率统计之间最重要的一座桥：

| 分布假设 | 负对数似然 | 等价损失 |
|----------|-----------|----------|
| 高斯噪声（回归） | $\sum\frac{(y_i - \hat y_i)^2}{2\sigma^2} + \text{const}$ | **MSE**（差常数） |
| 伯努利（二分类） | $-\sum\big[y_i\log p_i + (1-y_i)\log(1-p_i)\big]$ | **BCE** |
| 类别分布（多分类） | $-\sum_i \log p_{y_i}$ | **交叉熵 CE** |
| 拉普拉斯噪声（回归） | $\sum |y_i - \hat y_i| + \text{const}$ | **MAE** |

选择损失函数 = 选择你对数据噪声/输出形式的分布假设。这就是 08 课"概率综合"的伏笔。


In [ ]:
# 线性回归的 MLE 视角：y = w·x + 高斯噪声 → 最小二乘
rng = np.random.default_rng(1)
x = rng.uniform(-2, 2, 300)
w_true = 1.5
y = w_true*x + rng.normal(0, 0.3, 300)     # 高斯噪声

# 解析最小二乘（零截距）：ŵ = Σxy / Σx²
w_ols = np.sum(x*y) / np.sum(x**2)
print(f"最小二乘 ŵ = {w_ols:.4f}（真值 1.5）")

# 数值：梯度下降最小化 MSE（= 高斯 NLL 的一环）
wt = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
xt = torch.tensor(x, dtype=torch.float64); yt = torch.tensor(y, dtype=torch.float64)
opt = torch.optim.SGD([wt], lr=0.02)
for _ in range(500):
    opt.zero_grad()
    loss = ((xt*wt - yt)**2).mean()
    loss.backward(); opt.step()
print(f"梯度下降 ŵ = {wt.item():.4f}（与最小二乘一致 ✓）")


## 5. 过拟合的似然视角


模型越复杂（参数越多），通常越能提高训练集上的似然——极端情况下可以记忆每一个数据点。但这不等于泛化好：

- 训练似然 ↑ 但测试似然 ↓ → **过拟合**
- 正则化（nn-core 04）本质是在 NLL 上叠加参数的**先验**（MAP 估计：$\hat\theta = \arg\max \log p(x|\theta) + \log p(\theta)$）

"最大似然 + 先验 = 最大后验"，贝叶斯公式在模型选择上的直接应用。


## 课后练习


1. **完整推导**：写出高斯 NLL 对 $\sigma^2$ 求导并解出 $\hat\sigma^2$。
2. **伯努利 MLE**：对 $\ell(p) = \sum [y_i\log p + (1-y_i)\log(1-p)]$ 求导解出 $\hat p = \bar y$。
3. **数值实验**：$n=50$ 样本下重复 MLE，看 $\hat\mu$ 的分布（抽样分布），标准差应 ≈ $\sigma/\sqrt n$。
4. **MAP**：高斯先验 $\mu \sim N(0, \tau^2)$ 下推导 $\hat\mu_{MAP}$，说明它与 L2 正则的联系。
5. **思考**：为什么分类用 CE 而回归常用 MSE？噪声分布假设分别是什么？
